# Focused sketch → photo retraining (coarse-to-fine, resumable)

This notebook trains **only pencil sketch → colour photo**. It does not modify or
retrain the accepted photo → pencil model.

The selected design is deliberately conservative:

1. Freeze the strongest existing sketch→photo U-Net as a structural **coarse model**.
2. Train a smaller residual refiner on both real FS2K artist pairs and the persistent
   COCO 20K photo set.
3. Use paired reconstruction, LPIPS, edge, colour, feature-matching, and a slowly
   warmed conditional PatchGAN signal.
4. Keep an exponential-moving-average (EMA) copy and only accept checkpoints that
   pass domain-balanced validation. If refinement makes either domain substantially
   worse, the notebook keeps the earlier checkpoint.

## Honest expectations

- This is a stronger experiment, not a promise of perfect reconstruction. A monochrome
  sketch does not contain the original colours.
- Native training/output is 512 px. Upscaling later cannot invent verified detail.
- The notebook **never downloads COCO again**. It requires the existing persistent
  `datasets/coco2017_general_20k` folder in Drive.
- FS2K is persisted under Drive and is downloaded only once if it is not already there.
- Every 250 updates are saved. After a Colab limit/disconnection, reconnect and rerun
  all cells; training resumes from `last_training_state.pt`.

Use a GPU runtime, then run cells from top to bottom. Do not run the old master-training
cell at the same time.


In [ ]:
# Install only the small packages missing from a normal Colab GPU runtime.
# Do not replace Colab's torch/torchvision wheels.
import subprocess, sys
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'lpips==0.1.4', 'gdown==5.2.0',
])
print('Dependencies ready.')


In [ ]:
from pathlib import Path
from collections import defaultdict
from copy import deepcopy
import hashlib, json, math, os, random, shutil, time, zipfile

import cv2
import gdown
import lpips
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image, ImageEnhance, ImageFilter, ImageOps
from torch import nn
from torch.utils.data import DataLoader, Dataset, Subset
from tqdm.auto import tqdm

assert torch.cuda.is_available(), 'Choose Runtime → Change runtime type → GPU, then rerun.'
DEVICE = torch.device('cuda')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False

# Quality/compute controls. 8,000 updates are resumable and normally require several
# Colab sessions. Lowering this value reduces training, not just waiting time.
TOTAL_UPDATES = 8_000
START_512_AT = 4_000
SAVE_EVERY = 250
VALIDATE_EVERY = 500
FS2K_SAMPLE_PROBABILITY = 0.50
VALIDATION_PER_DOMAIN = 16
EARLY_STOPPING_VALIDATIONS = 10

from google.colab import drive
drive.mount('/content/drive')

SAVE_ROOT = Path('/content/drive/MyDrive/Sketch2Photo_training')
COCO_ROOT = SAVE_ROOT / 'datasets' / 'coco2017_general_20k'
FS2K_STORAGE = SAVE_ROOT / 'datasets' / 'FS2K_persistent'
RUN_ROOT = SAVE_ROOT / 'focused_sketch_to_photo_v2'
RUN_ROOT.mkdir(parents=True, exist_ok=True)
FS2K_STORAGE.mkdir(parents=True, exist_ok=True)

# Prefer the completed 20K model; fall back only when that exact checkpoint is absent.
COARSE_CANDIDATES = [
    SAVE_ROOT / 'general_20k_bidirectional_v1/general_rescue/unet_multiscale_patchgan/sketch_to_photo/best.pt',
    SAVE_ROOT / 'general_20k_sketch_photo_v1/general_rescue/unet_multiscale_patchgan/sketch_to_photo/best.pt',
    SAVE_ROOT / 'face_specific_rescue_v1/face_rescue/unet/sketch_to_photo/best.pt',
]
COARSE_CHECKPOINT = next((path for path in COARSE_CANDIDATES if path.is_file()), None)
assert COARSE_CHECKPOINT is not None, 'No completed sketch→photo U-Net checkpoint was found in Drive.'

print('GPU:', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), 'GiB')
print('Existing coarse model:', COARSE_CHECKPOINT)
print('New run folder:', RUN_ROOT)
print('COCO will not be downloaded by this notebook.')


## 1. Discover and audit the existing paired data

This cell fails instead of silently downloading COCO when the persistent 20K folder is
missing. FS2K is stored in Drive so a later Colab runtime can reuse it.


In [ ]:
ALLOWED_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}

def atomic_json(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(json.dumps(value, indent=2, default=str))
    os.replace(temporary, path)

def atomic_torch(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.tmp')
    torch.save(value, temporary)
    os.replace(temporary, path)

def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def safe_extract(zip_path, destination):
    destination = Path(destination); destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for entry in archive.infolist():
            resolved = (destination / entry.filename).resolve()
            if not resolved.is_relative_to(root) or ((entry.external_attr >> 16) & 0o170000) == 0o120000:
                raise ValueError('Unsafe ZIP entry: ' + entry.filename)
        archive.extractall(destination)

def find_fs2k_root():
    candidates = list(FS2K_STORAGE.rglob('anno_train.json'))
    if not candidates:
        archive = FS2K_STORAGE / 'FS2K.zip'
        if not archive.is_file():
            print('FS2K is absent from Drive; downloading it once to persistent storage...')
            gdown.download(
                id='1saIMhQ3dc5_ftkfGmBPbCluRn_zy7QQp',
                output=str(archive), quiet=False,
            )
        if not zipfile.is_zipfile(archive):
            raise ValueError('The downloaded FS2K file is not a valid ZIP.')
        safe_extract(archive, FS2K_STORAGE / 'extracted')
        candidates = list(FS2K_STORAGE.rglob('anno_train.json'))
    assert len(candidates) == 1, f'Expected one FS2K root, found {len(candidates)}.'
    root = candidates[0].parent
    assert (root / 'anno_test.json').is_file(), 'FS2K test annotations are missing.'
    return root

def normalize_stem(value):
    return Path(str(value).replace('\\', '/')).with_suffix('').as_posix().casefold()

def image_index(root, folder):
    base = root / folder
    index = defaultdict(list)
    for path in base.rglob('*'):
        if path.is_file() and path.suffix.lower() in ALLOWED_EXTENSIONS:
            index[normalize_stem(path.relative_to(base))].append(path)
    assert index, f'No images found in {base}'
    return index

def build_fs2k_records(root):
    train_rows = json.loads((root / 'anno_train.json').read_text())
    test_rows = json.loads((root / 'anno_test.json').read_text())
    grouped = defaultdict(list)
    for row in train_rows:
        grouped[str(row['style'])].append(row)
    tagged = []
    for style, rows in sorted(grouped.items()):
        rows = sorted(rows, key=lambda row: row['image_name'])
        random.Random(SEED + int(hashlib.sha256(style.encode()).hexdigest()[:8], 16)).shuffle(rows)
        cut = max(1, round(0.2 * len(rows)))
        tagged += [(row, 'val') for row in rows[:cut]]
        tagged += [(row, 'train') for row in rows[cut:]]
    tagged += [(row, 'test') for row in test_rows]

    photos, sketches = image_index(root, 'photo'), image_index(root, 'sketch')
    records, excluded = [], []
    for row, split in tagged:
        photo_stem = Path(str(row['image_name']).replace('\\', '/')).with_suffix('').as_posix()
        parts = photo_stem.split('/')
        parts[0] = parts[0].replace('photo', 'sketch', 1)
        sketch_path = Path('/'.join(parts))
        name = sketch_path.name
        if name.startswith('image'):
            name = 'sketch' + name[len('image'):]
        sketch_stem = (sketch_path.parent / name).as_posix()
        photo_matches = photos.get(normalize_stem(photo_stem), [])
        sketch_matches = sketches.get(normalize_stem(sketch_stem), [])
        if len(photo_matches) != 1 or len(sketch_matches) != 1:
            excluded.append(str(row['image_name']))
            continue
        records.append({
            'id': 'fs2k_' + photo_stem,
            'split': split,
            'domain': 'fs2k_artist',
            'photo': str(photo_matches[0]),
            'sketch': str(sketch_matches[0]),
        })
    assert len(records) >= int(0.98 * len(tagged)), (
        f'Only {len(records)}/{len(tagged)} FS2K pairs were resolved. Check the archive.'
    )
    return records, excluded

def build_coco_records():
    manifest_path = COCO_ROOT / 'manifest.json'
    photo_root = COCO_ROOT / 'photos'
    assert manifest_path.is_file(), (
        f'Missing {manifest_path}. Copy/share the completed dataset folder; this notebook will not redownload COCO.'
    )
    manifest = json.loads(manifest_path.read_text())
    records = []
    for row in manifest.get('records', []):
        filename = row.get('file_name') or Path(row.get('photo', '')).name
        photo = photo_root / filename
        if photo.is_file():
            records.append({
                'id': str(row.get('id', 'coco_' + photo.stem)),
                'split': row['split'],
                'domain': 'coco_synthetic',
                'photo': str(photo),
                'sketch': '',
            })
    train_count = sum(row['split'] == 'train' for row in records)
    assert train_count >= 20_000, (
        f'Only {train_count}/20000 persistent COCO training photos were found. '
        'Do not train on an incomplete folder.'
    )
    return records

FS2K_ROOT = find_fs2k_root()
FS2K_RECORDS, FS2K_EXCLUDED = build_fs2k_records(FS2K_ROOT)
COCO_RECORDS = build_coco_records()

FS2K_SPLITS = {split: [r for r in FS2K_RECORDS if r['split'] == split] for split in ('train','val','test')}
COCO_SPLITS = {split: [r for r in COCO_RECORDS if r['split'] == split] for split in ('train','val','test')}
for split in ('train', 'val', 'test'):
    print(split, '| FS2K:', len(FS2K_SPLITS[split]), '| COCO:', len(COCO_SPLITS[split]))

DATA_FINGERPRINT = hashlib.sha256(json.dumps({
    'fs2k': [(r['id'], r['split']) for r in FS2K_RECORDS],
    'coco': [(r['id'], r['split']) for r in COCO_RECORDS],
    'seed': SEED,
}, sort_keys=True).encode()).hexdigest()
atomic_json(RUN_ROOT / 'data_audit.json', {
    'fs2k_root': str(FS2K_ROOT),
    'fs2k_usable': len(FS2K_RECORDS),
    'fs2k_excluded': FS2K_EXCLUDED,
    'coco_root': str(COCO_ROOT),
    'coco_usable': len(COCO_RECORDS),
    'fingerprint': DATA_FINGERPRINT,
})
print('Data fingerprint:', DATA_FINGERPRINT)


## 2. Paired preprocessing

Real FS2K sketches remain real artist inputs. COCO photos receive a deterministic
graphite conversion only to create aligned training conditions; the original photo is
always the target. Geometry-changing augmentation is applied identically to both sides.


In [ ]:
def fit_canvas(image, size):
    image = ImageOps.exif_transpose(image).convert('RGB')
    scale = min(size / image.width, size / image.height)
    width, height = max(1, round(image.width * scale)), max(1, round(image.height * scale))
    resized = image.resize((width, height), Image.Resampling.LANCZOS)
    canvas = Image.new('RGB', (size, size), 'white')
    left, top = (size - width) // 2, (size - height) // 2
    canvas.paste(resized, (left, top))
    return canvas, (left, top, left + width, top + height)

def make_clean_line_art(photo):
    rgb = np.asarray(photo.convert('RGB'))
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    smooth = cv2.bilateralFilter(gray, 7, 32, 7)
    inverted = 255 - smooth
    blurred = cv2.GaussianBlur(inverted, (0, 0), 5.0)
    dodge = cv2.divide(smooth, np.maximum(255 - blurred, 1).astype(np.uint8), scale=256)
    fine = cv2.GaussianBlur(smooth, (0, 0), .65).astype(np.float32)
    medium = cv2.GaussianBlur(smooth, (0, 0), 1.65).astype(np.float32)
    edges = np.maximum(medium - fine, 0) / 255.0
    shadow = np.power((255 - smooth.astype(np.float32)) / 255.0, 1.45)
    density = .74 * ((255 - dodge.astype(np.float32)) / 255.0) + .22 * shadow + 2.1 * edges
    drawing = np.clip(254 - 255 * (1 - np.exp(-2.05 * density)), 8, 255).astype(np.uint8)
    return Image.fromarray(drawing).convert('RGB')

def augment_sketch(image, rng):
    sketch = ImageOps.autocontrast(image.convert('L'), cutoff=.2)
    sketch = ImageEnhance.Contrast(sketch).enhance(rng.uniform(.88, 1.15))
    sketch = ImageEnhance.Brightness(sketch).enhance(rng.uniform(.97, 1.03))
    if rng.random() < .15:
        sketch = sketch.filter(ImageFilter.MinFilter(3))
    elif rng.random() < .15:
        sketch = sketch.filter(ImageFilter.GaussianBlur(rng.uniform(.15, .45)))
    array = np.asarray(sketch, dtype=np.float32)
    noise = np.random.default_rng(rng.randrange(2**32)).normal(0, rng.uniform(0, 1.0), array.shape)
    return Image.fromarray(np.clip(array + noise, 0, 255).astype(np.uint8)).convert('RGB')

def tensor_image(image):
    array = np.asarray(image, dtype=np.float32).copy() / 127.5 - 1.0
    return torch.from_numpy(array.transpose(2, 0, 1))

def load_pair(record, resolution):
    with Image.open(record['photo']) as image:
        photo, box = fit_canvas(image, resolution)
    if record['domain'] == 'fs2k_artist':
        with Image.open(record['sketch']) as image:
            sketch, sketch_box = fit_canvas(image, resolution)
        assert sketch_box == box, 'FS2K pair geometry does not match.'
        sketch = sketch.convert('L').convert('RGB')
    else:
        sketch = make_clean_line_art(photo)
    return photo, sketch, box

class FocusedDataset(Dataset):
    def __init__(self, total_indices, resolution, training=True):
        self.total_indices = total_indices
        self.resolution = resolution
        self.training = training

    def __len__(self):
        return len(self.total_indices)

    def __getitem__(self, position):
        global_index = self.total_indices[position]
        rng = random.Random(SEED + 1_000_003 * global_index)
        use_fs2k = rng.random() < FS2K_SAMPLE_PROBABILITY
        pool = FS2K_SPLITS['train'] if use_fs2k else COCO_SPLITS['train']
        record = pool[rng.randrange(len(pool))]
        photo, sketch, _ = load_pair(record, self.resolution)
        if self.training and rng.random() < .5:
            photo, sketch = ImageOps.mirror(photo), ImageOps.mirror(sketch)
        if self.training and rng.random() < .35:
            sketch = augment_sketch(sketch, rng)
        return {
            'source': tensor_image(sketch),
            'target': tensor_image(photo),
            'domain': record['domain'],
            'id': record['id'],
        }

VALIDATION_RECORDS = (
    FS2K_SPLITS['val'][:VALIDATION_PER_DOMAIN]
    + COCO_SPLITS['val'][:VALIDATION_PER_DOMAIN]
)
assert len(VALIDATION_RECORDS) == 2 * VALIDATION_PER_DOMAIN

fig, axes = plt.subplots(4, 2, figsize=(9, 15))
preview_records = FS2K_SPLITS['train'][:2] + COCO_SPLITS['train'][:2]
for row, record in enumerate(preview_records):
    photo, sketch, box = load_pair(record, 512)
    for ax, image, title in zip(axes[row], [sketch.crop(box), photo.crop(box)], ['Sketch input', 'Photo target']):
        ax.imshow(image); ax.set_title(record['domain'] + ' — ' + title); ax.axis('off')
plt.tight_layout()
plt.savefig(RUN_ROOT / 'pair_audit.png', dpi=130)
plt.show()
print('STOP if any row above is mismatched. Otherwise continue.')


## 3. Coarse-to-fine model

The old U-Net is frozen, so its learned geometry cannot be erased. The new trainable
refiner sees both the sketch and coarse colour result and predicts a bounded residual.
The discriminator is training-only and is not required for inference.


In [ ]:
def unet_block(input_channels, output_channels):
    return nn.Sequential(
        nn.Conv2d(input_channels, output_channels, 3, padding=1),
        nn.GroupNorm(8, output_channels), nn.SiLU(),
        nn.Conv2d(output_channels, output_channels, 3, padding=1),
        nn.GroupNorm(8, output_channels), nn.SiLU(),
    )

class PencilUNet(nn.Module):
    def __init__(self, width=64, output_mode='direct'):
        super().__init__()
        self.output_mode = output_mode
        self.enc = nn.ModuleList([
            unet_block(3, width), unet_block(width, width*2),
            unet_block(width*2, width*4), unet_block(width*4, width*8),
        ])
        self.mid = unet_block(width*8, width*8)
        self.dec = nn.ModuleList([
            unet_block(width*16, width*4), unet_block(width*8, width*2),
            unet_block(width*4, width), unet_block(width*2, width),
        ])
        self.out = nn.Conv2d(width, 3, 1)

    def forward(self, x):
        source, skips = x, []
        for layer in self.enc:
            x = layer(x); skips.append(x); x = F.avg_pool2d(x, 2)
        x = self.mid(x)
        for layer, skip in zip(self.dec, reversed(skips)):
            x = F.interpolate(x, size=skip.shape[-2:], mode='bilinear', align_corners=False)
            x = layer(torch.cat([x, skip], 1))
        prediction = self.out(x)
        return (source + prediction).clamp(-1, 1) if self.output_mode == 'residual' else torch.tanh(prediction)

class RefineBlock(nn.Module):
    def __init__(self, input_channels, output_channels):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(input_channels, output_channels, 3, padding=1),
            nn.GroupNorm(8, output_channels), nn.SiLU(),
            nn.Conv2d(output_channels, output_channels, 3, padding=1),
            nn.GroupNorm(8, output_channels), nn.SiLU(),
        )
    def forward(self, x):
        return self.layers(x)

class CoarseToFineRefiner(nn.Module):
    def __init__(self, width=32):
        super().__init__()
        self.enc1 = RefineBlock(6, width)
        self.enc2 = RefineBlock(width, width*2)
        self.enc3 = RefineBlock(width*2, width*4)
        self.mid = RefineBlock(width*4, width*8)
        self.dec3 = RefineBlock(width*8 + width*4, width*4)
        self.dec2 = RefineBlock(width*4 + width*2, width*2)
        self.dec1 = RefineBlock(width*2 + width, width)
        self.out = nn.Conv2d(width, 3, 1)
        nn.init.zeros_(self.out.weight); nn.init.zeros_(self.out.bias)

    def forward(self, sketch, coarse):
        x1 = self.enc1(torch.cat([sketch, coarse], 1))
        x2 = self.enc2(F.avg_pool2d(x1, 2))
        x3 = self.enc3(F.avg_pool2d(x2, 2))
        mid = self.mid(F.avg_pool2d(x3, 2))
        x = F.interpolate(mid, size=x3.shape[-2:], mode='bilinear', align_corners=False)
        x = self.dec3(torch.cat([x, x3], 1))
        x = F.interpolate(x, size=x2.shape[-2:], mode='bilinear', align_corners=False)
        x = self.dec2(torch.cat([x, x2], 1))
        x = F.interpolate(x, size=x1.shape[-2:], mode='bilinear', align_corners=False)
        delta = self.out(self.dec1(torch.cat([x, x1], 1)))
        # Bounded residual in logit-like tanh space avoids clipping gradients.
        base = torch.atanh(coarse.clamp(-.995, .995))
        return torch.tanh(base + .45 * delta)

class FeaturePatchDiscriminator(nn.Module):
    def __init__(self, width=48):
        super().__init__()
        channels = [6, width, width*2, width*4, width*8]
        self.blocks = nn.ModuleList()
        for index in range(4):
            stride = 2 if index < 3 else 1
            self.blocks.append(nn.Sequential(
                nn.utils.spectral_norm(nn.Conv2d(channels[index], channels[index+1], 4, stride, 1)),
                nn.LeakyReLU(.2, inplace=True),
            ))
        self.out = nn.utils.spectral_norm(nn.Conv2d(channels[-1], 1, 4, 1, 1))

    def forward(self, paired):
        features = []
        for block in self.blocks:
            paired = block(paired); features.append(paired)
        return self.out(paired), features

coarse_state = torch.load(COARSE_CHECKPOINT, map_location='cpu', weights_only=True)
assert coarse_state.get('direction') == 'sketch_to_photo'
coarse_width = int(coarse_state.get('width', 64))
coarse_mode = coarse_state.get('output_mode', 'direct')
coarse = PencilUNet(coarse_width, coarse_mode).to(DEVICE)
coarse.load_state_dict(coarse_state['generator'], strict=True)
coarse.eval().requires_grad_(False)

refiner = CoarseToFineRefiner(32).to(DEVICE)
ema_refiner = deepcopy(refiner).eval().requires_grad_(False)
discriminator_full = FeaturePatchDiscriminator().to(DEVICE)
discriminator_half = FeaturePatchDiscriminator().to(DEVICE)

print('Loaded coarse checkpoint step:', coarse_state.get('step'))
print('Coarse output mode:', coarse_mode)
print('Trainable refiner parameters:', sum(p.numel() for p in refiner.parameters()))
print('Initial refiner output is exactly the frozen coarse output.')


## 4. Validation and losses

Validation weights FS2K and COCO equally. “Best” is selected without looking at the test
set. A domain guard prevents a small average gain from hiding major damage to portraits or
general scenes.


In [ ]:
PERCEPTUAL = lpips.LPIPS(net='alex', version='0.1').to(DEVICE).eval().requires_grad_(False)

def spatial_gradients(x):
    return x[:, :, :, 1:] - x[:, :, :, :-1], x[:, :, 1:, :] - x[:, :, :-1, :]

def colour_statistics(x):
    return x.mean((2,3)), x.std((2,3), unbiased=False)

def charbonnier(prediction, target, epsilon=1e-3):
    return torch.sqrt((prediction - target).square() + epsilon**2).mean()

def reconstruction_loss(fake, target, sketch):
    fake_dx, fake_dy = spatial_gradients(fake)
    target_dx, target_dy = spatial_gradients(target)
    fake_mean, fake_std = colour_statistics(fake)
    target_mean, target_std = colour_statistics(target)
    # LPIPS backpropagation is kept in float32. On T4, float16 LPIPS gradients can
    # overflow even when the displayed scalar loss itself is finite.
    with torch.autocast('cuda', enabled=False):
        perceptual = PERCEPTUAL(fake.float(), target.float()).mean()
    # Low weights prevent the new texture objective from overwhelming paired fidelity.
    return (
        5.0 * charbonnier(fake, target)
        + 1.5 * perceptual
        + 1.25 * (charbonnier(fake_dx, target_dx) + charbonnier(fake_dy, target_dy))
        + .60 * (F.l1_loss(fake_mean, target_mean) + F.l1_loss(fake_std, target_std))
    )

@torch.no_grad()
def predict_tensor(sketch_tensor, active_refiner):
    coarse_result = coarse(sketch_tensor)
    return coarse_result, active_refiner(sketch_tensor, coarse_result)

@torch.no_grad()
def validate(active_refiner):
    active_refiner.eval(); rows = []
    for record in tqdm(VALIDATION_RECORDS, desc='Validation', leave=False):
        photo, sketch, _ = load_pair(record, 512)
        source = tensor_image(sketch)[None].to(DEVICE)
        target = tensor_image(photo)[None].to(DEVICE)
        with torch.autocast('cuda', dtype=torch.float16):
            _, prediction = predict_tensor(source, active_refiner)
            lpips_value = float(PERCEPTUAL(prediction, target).mean())
        mae = float(F.l1_loss(prediction.float(), target.float()) / 2)
        pred_dx, pred_dy = spatial_gradients(prediction.float())
        target_dx, target_dy = spatial_gradients(target.float())
        edge = float((F.l1_loss(pred_dx, target_dx) + F.l1_loss(pred_dy, target_dy)) / 4)
        rows.append({'id': record['id'], 'domain': record['domain'],
                     'lpips': lpips_value, 'mae': mae, 'edge': edge})
    summary = {}
    for domain in ('fs2k_artist', 'coco_synthetic'):
        domain_rows = [row for row in rows if row['domain'] == domain]
        summary[domain] = {
            metric: float(np.mean([row[metric] for row in domain_rows]))
            for metric in ('lpips', 'mae', 'edge')
        }
    summary['balanced_score'] = float(np.mean([
        values['lpips'] + .25 * values['mae'] + .10 * values['edge']
        for values in summary.values()
    ]))
    return summary, rows

def tensor_to_image(tensor):
    array = ((tensor.detach().float().cpu().clamp(-1,1)[0].permute(1,2,0).numpy()+1)*127.5)
    return Image.fromarray(array.round().astype('uint8'))

def render_gate(active_refiner, step, filename):
    active_refiner.eval()
    records = FS2K_SPLITS['val'][:3] + COCO_SPLITS['val'][:3]
    fig, axes = plt.subplots(len(records), 4, figsize=(14, 3.7*len(records)))
    for row, record in enumerate(records):
        photo, sketch, box = load_pair(record, 512)
        source = tensor_image(sketch)[None].to(DEVICE)
        with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
            coarse_result, refined = predict_tensor(source, active_refiner)
        images = [sketch.crop(box), tensor_to_image(coarse_result).crop(box),
                  tensor_to_image(refined).crop(box), photo.crop(box)]
        titles = ['Input sketch', 'Frozen coarse U-Net', f'Refined step {step}', 'Real target']
        for axis, image, title in zip(axes[row], images, titles):
            axis.imshow(image); axis.set_title(title); axis.axis('off')
    fig.tight_layout(); fig.savefig(RUN_ROOT / filename, dpi=130); plt.show(); plt.close(fig)


## 5. Train or resume

Run this cell. If Colab disconnects, reconnect and rerun all preceding cells, then this
cell. It resumes automatically. The first validation at step 0 records the frozen U-Net
baseline, so later checkpoints must beat a real baseline rather than merely finish.


In [ ]:
RECIPE = {
    'format': 'sketch2photo-focused-refiner-v1',
    'seed': SEED,
    'total_updates': TOTAL_UPDATES,
    'start_512_at': START_512_AT,
    'save_every': SAVE_EVERY,
    'validate_every': VALIDATE_EVERY,
    'fs2k_probability': FS2K_SAMPLE_PROBABILITY,
    'coarse_checkpoint': str(COARSE_CHECKPOINT),
    'coarse_step': int(coarse_state.get('step', -1)),
    'data_fingerprint': DATA_FINGERPRINT,
    'generator_lr': 3e-5,
    'discriminator_lr': 6e-5,
    'adversarial_max': .05,
    'adversarial_warmup': 2_000,
    'implementation': 'v2-fp32-lpips-safe-amp',
}
SIGNATURE = hashlib.sha256(json.dumps(RECIPE, sort_keys=True).encode()).hexdigest()
atomic_json(RUN_ROOT / 'recipe.json', RECIPE | {'signature': SIGNATURE})

optimizer_g = torch.optim.AdamW(refiner.parameters(), lr=RECIPE['generator_lr'], betas=(.5,.999), weight_decay=1e-4)
discriminator_parameters = list(discriminator_full.parameters()) + list(discriminator_half.parameters())
optimizer_d = torch.optim.AdamW(discriminator_parameters, lr=RECIPE['discriminator_lr'], betas=(.5,.999))
scheduler_g = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_g, T_max=TOTAL_UPDATES, eta_min=3e-6)
scheduler_d = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_d, T_max=TOTAL_UPDATES, eta_min=6e-6)
# A conservative initial scale prevents first-step T4 overflow.
scaler = torch.amp.GradScaler('cuda', init_scale=256.0, growth_factor=2.0, backoff_factor=.5, growth_interval=2_000)

progress = {
    'step': 0, 'best_step': 0, 'best_score': float('inf'),
    'bad_validations': 0, 'seconds': 0.0, 'baseline': None,
}
history = []
last_path = RUN_ROOT / 'last_training_state.pt'

if last_path.is_file():
    state = torch.load(last_path, map_location='cpu', weights_only=False)
    assert state['signature'] == SIGNATURE, (
        'Training configuration changed. Restore the earlier values or choose a new RUN_ROOT.'
    )
    refiner.load_state_dict(state['refiner'])
    ema_refiner.load_state_dict(state['ema_refiner'])
    discriminator_full.load_state_dict(state['discriminator_full'])
    discriminator_half.load_state_dict(state['discriminator_half'])
    optimizer_g.load_state_dict(state['optimizer_g']); optimizer_d.load_state_dict(state['optimizer_d'])
    scheduler_g.load_state_dict(state['scheduler_g']); scheduler_d.load_state_dict(state['scheduler_d'])
    scaler.load_state_dict(state['scaler'])
    progress, history = state['progress'], state['history']
    torch.set_rng_state(state['torch_rng']); torch.cuda.set_rng_state_all(state['cuda_rng'])
    random.setstate(state['python_rng']); np.random.set_state(state['numpy_rng'])
    print('Resumed safely at update', progress['step'])

def cpu_state_dict(model):
    return {key: value.detach().cpu() for key, value in model.state_dict().items()}

def save_deploy_checkpoint(path, summary):
    atomic_torch(path, {
        'format': 'sketch2photo-focused-refiner-v1',
        'direction': 'sketch_to_photo',
        'resolution': 512,
        'step': progress['step'],
        'validation': summary,
        'coarse_width': coarse_width,
        'coarse_output_mode': coarse_mode,
        'coarse_generator': cpu_state_dict(coarse),
        'refiner_width': 32,
        'ema_refiner': cpu_state_dict(ema_refiner),
        'data_fingerprint': DATA_FINGERPRINT,
        'recipe_signature': SIGNATURE,
    })

def save_refiner_snapshot(path, summary):
    # Intermediate snapshots omit the unchanged 51 MB coarse U-Net. The selected
    # best.pt remains self-contained, while Drive storage stays reasonable.
    atomic_torch(path, {
        'format': 'sketch2photo-focused-refiner-snapshot-v1',
        'direction': 'sketch_to_photo',
        'resolution': 512,
        'step': progress['step'],
        'validation': summary,
        'refiner_width': 32,
        'ema_refiner': cpu_state_dict(ema_refiner),
        'coarse_checkpoint': str(COARSE_CHECKPOINT),
        'recipe_signature': SIGNATURE,
    })

started = time.perf_counter(); elapsed_before = progress['seconds']
def save_last():
    progress['seconds'] = elapsed_before + time.perf_counter() - started
    atomic_torch(last_path, {
        'signature': SIGNATURE,
        'refiner': refiner.state_dict(), 'ema_refiner': ema_refiner.state_dict(),
        'discriminator_full': discriminator_full.state_dict(),
        'discriminator_half': discriminator_half.state_dict(),
        'optimizer_g': optimizer_g.state_dict(), 'optimizer_d': optimizer_d.state_dict(),
        'scheduler_g': scheduler_g.state_dict(), 'scheduler_d': scheduler_d.state_dict(),
        'scaler': scaler.state_dict(), 'progress': dict(progress), 'history': history,
        'torch_rng': torch.get_rng_state(), 'cuda_rng': torch.cuda.get_rng_state_all(),
        'python_rng': random.getstate(), 'numpy_rng': np.random.get_state(),
    })
    atomic_json(RUN_ROOT / 'history.json', history)
    atomic_json(RUN_ROOT / 'progress.json', progress)

def run_validation():
    summary, rows = validate(ema_refiner)
    entry = {'type':'validation', 'step':progress['step'], **summary}
    history.append(entry)
    atomic_json(RUN_ROOT / f"validation_{progress['step']:07d}.json", {'summary':summary, 'rows':rows})
    if progress['baseline'] is None:
        progress['baseline'] = summary
    baseline = progress['baseline']
    domain_guard = all(
        summary[domain]['lpips'] <= baseline[domain]['lpips'] * 1.10
        and summary[domain]['mae'] <= baseline[domain]['mae'] * 1.10
        for domain in ('fs2k_artist','coco_synthetic')
    )
    improved = domain_guard and summary['balanced_score'] < progress['best_score'] - 1e-5
    if improved:
        progress['best_score'] = summary['balanced_score']; progress['best_step'] = progress['step']
        progress['bad_validations'] = 0
        save_deploy_checkpoint(RUN_ROOT / 'best.pt', summary)
    else:
        progress['bad_validations'] += 1
    save_refiner_snapshot(RUN_ROOT / f"snapshot_{progress['step']:07d}.pt", summary)
    render_gate(ema_refiner, progress['step'], f"gate_{progress['step']:07d}.png")
    print('Validation:', json.dumps(summary, indent=2))
    print('Accepted as best:', improved, '| best step:', progress['best_step'])

if not history:
    run_validation(); save_last()

bar = tqdm(total=TOTAL_UPDATES, initial=progress['step'], desc='Focused sketch→photo')
stop_early = False
consecutive_nonfinite_gradients = 0
while progress['step'] < TOTAL_UPDATES and not stop_early:
    resolution = 256 if progress['step'] < START_512_AT else 512
    boundary = min(TOTAL_UPDATES, START_512_AT if resolution == 256 else TOTAL_UPDATES)
    indices = list(range(progress['step'], boundary))
    dataset = FocusedDataset(indices, resolution, training=True)
    loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=2,
                        pin_memory=True, persistent_workers=len(dataset) > 2)
    refiner.train(); discriminator_full.train(); discriminator_half.train()

    for batch in loader:
        sketch = batch['source'].to(DEVICE, non_blocking=True)
        target = batch['target'].to(DEVICE, non_blocking=True)
        with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16):
            coarse_result = coarse(sketch)
        with torch.autocast('cuda', dtype=torch.float16):
            fake = refiner(sketch, coarse_result)

        # Conditional multi-scale discriminator update.
        optimizer_d.zero_grad(set_to_none=True)
        with torch.autocast('cuda', dtype=torch.float16):
            real_full, _ = discriminator_full(torch.cat([sketch, target], 1))
            fake_full, _ = discriminator_full(torch.cat([sketch, fake.detach()], 1))
            sketch_half = F.interpolate(sketch, scale_factor=.5, mode='bilinear', align_corners=False)
            target_half = F.interpolate(target, scale_factor=.5, mode='bilinear', align_corners=False)
            fake_half = F.interpolate(fake.detach(), scale_factor=.5, mode='bilinear', align_corners=False)
            real_half, _ = discriminator_half(torch.cat([sketch_half, target_half], 1))
            generated_half, _ = discriminator_half(torch.cat([sketch_half, fake_half], 1))
            loss_d = .25 * (
                F.relu(1-real_full).mean() + F.relu(1+fake_full).mean()
                + F.relu(1-real_half).mean() + F.relu(1+generated_half).mean()
            )
        scaler.scale(loss_d).backward()
        scaler.unscale_(optimizer_d); nn.utils.clip_grad_norm_(discriminator_parameters, 1.0)
        scaler.step(optimizer_d)

        # Refiner update. Feature matching stabilizes texture learning.
        for discriminator in (discriminator_full, discriminator_half):
            discriminator.requires_grad_(False)
        optimizer_g.zero_grad(set_to_none=True)
        with torch.autocast('cuda', dtype=torch.float16):
            fake_score_full, fake_features_full = discriminator_full(torch.cat([sketch, fake], 1))
            with torch.no_grad():
                _, real_features_full = discriminator_full(torch.cat([sketch, target], 1))
            refined_half = F.interpolate(fake, scale_factor=.5, mode='bilinear', align_corners=False)
            fake_score_half, fake_features_half = discriminator_half(torch.cat([sketch_half, refined_half], 1))
            with torch.no_grad():
                _, real_features_half = discriminator_half(torch.cat([sketch_half, target_half], 1))
            paired = reconstruction_loss(fake, target, sketch)
            feature_matching = sum(
                F.l1_loss(generated, real.detach())
                for generated, real in zip(fake_features_full + fake_features_half,
                                           real_features_full + real_features_half)
            ) / (len(fake_features_full) + len(fake_features_half))
            adversarial = -.5 * (fake_score_full.mean() + fake_score_half.mean())
            next_step = progress['step'] + 1
            adversarial_weight = RECIPE['adversarial_max'] * min(1.0, next_step / RECIPE['adversarial_warmup'])
            loss_g = paired + .75 * feature_matching + adversarial_weight * adversarial
        if not torch.isfinite(loss_g) or not torch.isfinite(loss_d):
            raise FloatingPointError('Non-finite training loss; last Drive checkpoint is still safe.')
        scaler.scale(loss_g).backward()
        scaler.unscale_(optimizer_g)
        gradient_norm = nn.utils.clip_grad_norm_(refiner.parameters(), 1.0, error_if_nonfinite=False)
        if not torch.isfinite(gradient_norm):
            # Do not corrupt the model or count a fake update. Reduce the scale and
            # move to a fresh batch. Repeated failures stop with the Drive state safe.
            optimizer_g.zero_grad(set_to_none=True)
            for discriminator in (discriminator_full, discriminator_half):
                discriminator.requires_grad_(True)
            consecutive_nonfinite_gradients += 1
            reduced_scale = max(float(scaler.get_scale()) * .5, 1.0)
            scaler.update(new_scale=reduced_scale)
            print(f'Skipped non-finite gradient batch; AMP scale reduced to {reduced_scale:g}.')
            if consecutive_nonfinite_gradients >= 3:
                save_last()
                raise FloatingPointError(
                    'Three consecutive non-finite gradient batches. The last Drive checkpoint is safe.'
                )
            continue
        consecutive_nonfinite_gradients = 0
        scaler.step(optimizer_g); scaler.update()
        for discriminator in (discriminator_full, discriminator_half):
            discriminator.requires_grad_(True)
        scheduler_g.step(); scheduler_d.step()

        with torch.no_grad():
            decay = .995 if next_step < 1_000 else .999
            for ema_parameter, parameter in zip(ema_refiner.parameters(), refiner.parameters()):
                ema_parameter.lerp_(parameter, 1-decay)

        progress['step'] = next_step
        history.append({
            'type':'train', 'step':next_step, 'resolution':resolution,
            'generator_loss':float(loss_g.detach()), 'paired_loss':float(paired.detach()),
            'feature_matching':float(feature_matching.detach()),
            'adversarial_loss':float(adversarial.detach()),
            'adversarial_weight':adversarial_weight,
            'discriminator_loss':float(loss_d.detach()),
            'generator_lr':scheduler_g.get_last_lr()[0],
        })
        bar.update(1); bar.set_postfix(res=resolution, g=round(float(loss_g.detach()),3), d=round(float(loss_d.detach()),3))

        if next_step % VALIDATE_EVERY == 0 or next_step == TOTAL_UPDATES:
            run_validation()
            if (next_step >= 3_000 and progress['bad_validations'] >= EARLY_STOPPING_VALIDATIONS):
                print('Early stopping: validation has not improved. The best checkpoint is preserved.')
                stop_early = True
        if next_step % SAVE_EVERY == 0 or next_step == TOTAL_UPDATES or stop_early:
            save_last()
        if next_step == boundary or stop_early:
            break
    del loader, dataset
    gc_collected = __import__('gc').collect(); torch.cuda.empty_cache()

bar.close(); save_last()
print('Training stopped at:', progress['step'])
print('Validation-selected best step:', progress['best_step'])
print('Best model:', RUN_ROOT / 'best.pt')


## 6. Teacher-facing learning curves and held-out error analysis

Image-to-image regression has no honest class “accuracy” percentage. Report the following
instead:

- **LPIPS**: perceptual difference; lower is better.
- **MAE**: average absolute pixel error; lower is better.
- **PSNR**: signal-to-error ratio in decibels; higher is better.
- **SSIM**: structural similarity from 0 to 1; higher is better.

The plots below show optimization/validation behaviour and compare the frozen coarse U-Net
against the validation-selected refiner on the untouched test split. Model selection never
uses these test scores. `TEST_LIMIT_PER_DOMAIN=None` evaluates every available test record;
set an integer only for a clearly labelled preliminary subset.


In [ ]:
import csv

BEST_PATH = RUN_ROOT / 'best.pt'
assert BEST_PATH.is_file(), 'Run training first; best.pt does not exist.'
best_state = torch.load(BEST_PATH, map_location='cpu', weights_only=True)
ema_refiner.load_state_dict(best_state['ema_refiner'], strict=True)
ema_refiner.to(DEVICE).eval()

REPORT_ROOT = RUN_ROOT / 'teacher_report'
REPORT_ROOT.mkdir(exist_ok=True)
TEST_LIMIT_PER_DOMAIN = None  # None = full held-out test; integer = labelled subset

# ---- Training and validation curves ----
saved_history = json.loads((RUN_ROOT / 'history.json').read_text())
train_rows = [row for row in saved_history if row.get('type') == 'train']
validation_rows = [row for row in saved_history if row.get('type') == 'validation']

def smoothed(rows, key, window=100):
    x = np.array([row['step'] for row in rows], dtype=float)
    y = np.array([row[key] for row in rows], dtype=float)
    if len(y) < window:
        return x, y
    kernel = np.ones(window) / window
    return x[window-1:], np.convolve(y, kernel, mode='valid')

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for key, label in [('generator_loss','Generator'), ('paired_loss','Paired reconstruction')]:
    x, y = smoothed(train_rows, key)
    axes[0,0].plot(x, y, label=label)
axes[0,0].set(title='Training losses (100-update moving average)', xlabel='Updates', ylabel='Loss')
axes[0,0].legend(); axes[0,0].grid(alpha=.25)

x, y = smoothed(train_rows, 'discriminator_loss')
axes[0,1].plot(x, y, color='tab:red')
axes[0,1].set(title='Discriminator loss (100-update moving average)', xlabel='Updates', ylabel='Hinge loss')
axes[0,1].grid(alpha=.25)

steps = [row['step'] for row in validation_rows]
axes[1,0].plot(steps, [row['fs2k_artist']['lpips'] for row in validation_rows], marker='o', label='FS2K portraits')
axes[1,0].plot(steps, [row['coco_synthetic']['lpips'] for row in validation_rows], marker='o', label='COCO scenes')
axes[1,0].set(title='Validation LPIPS — lower is better', xlabel='Updates', ylabel='LPIPS')
axes[1,0].legend(); axes[1,0].grid(alpha=.25)

axes[1,1].plot(steps, [row['balanced_score'] for row in validation_rows], marker='o', color='tab:purple')
axes[1,1].axvline(best_state['step'], color='tab:green', linestyle='--', label=f"Selected step {best_state['step']}")
axes[1,1].set(title='Domain-balanced validation score', xlabel='Updates', ylabel='Score (lower is better)')
axes[1,1].legend(); axes[1,1].grid(alpha=.25)
fig.tight_layout(); fig.savefig(REPORT_ROOT / 'training_validation_curves.png', dpi=160)
plt.show(); plt.close(fig)

# ---- Full-reference metrics on untouched test records ----
def uniform_ssim(prediction_01, target_01, window=11):
    mu_x = F.avg_pool2d(prediction_01, window, 1)
    mu_y = F.avg_pool2d(target_01, window, 1)
    sigma_x = F.avg_pool2d(prediction_01.square(), window, 1) - mu_x.square()
    sigma_y = F.avg_pool2d(target_01.square(), window, 1) - mu_y.square()
    sigma_xy = F.avg_pool2d(prediction_01 * target_01, window, 1) - mu_x * mu_y
    c1, c2 = .01**2, .03**2
    value = ((2*mu_x*mu_y+c1) * (2*sigma_xy+c2)) / ((mu_x.square()+mu_y.square()+c1) * (sigma_x+sigma_y+c2))
    return value.mean()

def metric_row(prediction, target, model_name, record, box):
    left, top, right, bottom = box
    prediction = prediction[:, :, top:bottom, left:right].float().clamp(-1,1)
    target = target[:, :, top:bottom, left:right].float().clamp(-1,1)
    if min(prediction.shape[-2:]) < 64:
        raise ValueError('content crop is too narrow for reliable LPIPS')
    with torch.autocast('cuda', dtype=torch.float16):
        perceptual = float(PERCEPTUAL(prediction, target).mean())
    prediction_01, target_01 = (prediction+1)/2, (target+1)/2
    mae = float(F.l1_loss(prediction_01, target_01))
    mse = float(F.mse_loss(prediction_01, target_01))
    psnr = float(10 * math.log10(1 / max(mse, 1e-12)))
    ssim = float(uniform_ssim(prediction_01, target_01))
    return {'id':record['id'], 'domain':record['domain'], 'model':model_name,
            'lpips':perceptual, 'mae':mae, 'psnr_db':psnr, 'ssim':ssim}

test_records = []
for domain_rows in (FS2K_SPLITS['test'], COCO_SPLITS['test']):
    rows = domain_rows if TEST_LIMIT_PER_DOMAIN is None else domain_rows[:TEST_LIMIT_PER_DOMAIN]
    test_records.extend(rows)

test_rows, excluded_rows = [], []
for record in tqdm(test_records, desc='Held-out test evaluation'):
    try:
        photo, sketch, box = load_pair(record, 512)
        source = tensor_image(sketch)[None].to(DEVICE)
        target = tensor_image(photo)[None].to(DEVICE)
        with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
            baseline, refined = predict_tensor(source, ema_refiner)
        test_rows.append(metric_row(baseline, target, 'Frozen coarse U-Net', record, box))
        test_rows.append(metric_row(refined, target, 'Selected refiner', record, box))
    except Exception as error:
        excluded_rows.append({'id':record['id'], 'domain':record['domain'], 'reason':str(error)})

assert test_rows, 'No test records were evaluated.'
scope = 'full_test' if TEST_LIMIT_PER_DOMAIN is None else f'subset_{TEST_LIMIT_PER_DOMAIN}_per_domain'
summary = []
for model_name in ('Frozen coarse U-Net', 'Selected refiner'):
    for domain in ('fs2k_artist', 'coco_synthetic'):
        rows = [row for row in test_rows if row['model']==model_name and row['domain']==domain]
        assert rows, f'No evaluated rows for {model_name}, {domain}'
        summary.append({
            'scope':scope, 'model':model_name, 'domain':domain, 'n':len(rows),
            **{metric:float(np.mean([row[metric] for row in rows]))
               for metric in ('lpips','mae','psnr_db','ssim')},
        })

with (REPORT_ROOT / 'test_metrics_per_image.csv').open('w', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=list(test_rows[0]))
    writer.writeheader(); writer.writerows(test_rows)
with (REPORT_ROOT / 'test_summary.csv').open('w', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=list(summary[0]))
    writer.writeheader(); writer.writerows(summary)
atomic_json(REPORT_ROOT / 'test_summary.json', {'scope':scope, 'summary':summary, 'excluded':excluded_rows})

print('Held-out evaluation scope:', scope)
for row in summary:
    print(row)
if excluded_rows:
    print('Excluded records (with reasons):', len(excluded_rows))

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
metric_specs = [('lpips','LPIPS ↓'), ('mae','MAE ↓'), ('psnr_db','PSNR dB ↑'), ('ssim','SSIM ↑')]
models = ['Frozen coarse U-Net', 'Selected refiner']
domains = ['fs2k_artist', 'coco_synthetic']
for axis, (metric, title) in zip(axes.flat, metric_specs):
    x = np.arange(len(domains)); width = .36
    for index, model_name in enumerate(models):
        values = [next(row[metric] for row in summary if row['model']==model_name and row['domain']==domain) for domain in domains]
        axis.bar(x + (index-.5)*width, values, width, label=model_name)
    axis.set_xticks(x, ['FS2K portraits','COCO scenes']); axis.set_title(title); axis.grid(axis='y', alpha=.2)
axes[0,0].legend()
fig.suptitle('Held-out test comparison — ' + scope)
fig.tight_layout(); fig.savefig(REPORT_ROOT / 'test_metric_comparison.png', dpi=160)
plt.show(); plt.close(fig)

# ---- Worst-case error examples for honest failure analysis ----
refined_rows = [row for row in test_rows if row['model']=='Selected refiner']
worst = sorted(refined_rows, key=lambda row:row['lpips'], reverse=True)[:6]
record_lookup = {record['id']:record for record in test_records}
fig, axes = plt.subplots(len(worst), 4, figsize=(14, 3.6*len(worst)))
for row_index, metric in enumerate(worst):
    record = record_lookup[metric['id']]
    photo, sketch, box = load_pair(record, 512)
    source = tensor_image(sketch)[None].to(DEVICE)
    with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
        baseline, refined = predict_tensor(source, ema_refiner)
    images = [sketch.crop(box), tensor_to_image(baseline).crop(box),
              tensor_to_image(refined).crop(box), photo.crop(box)]
    titles = ['Input sketch','Coarse U-Net',f"Refined — LPIPS {metric['lpips']:.3f}",'Real target']
    for axis, image, title in zip(axes[row_index], images, titles):
        axis.imshow(image); axis.set_title(title); axis.axis('off')
fig.tight_layout(); fig.savefig(REPORT_ROOT / 'worst_test_failures.png', dpi=150)
plt.show(); plt.close(fig)

print('Teacher report folder:', REPORT_ROOT)
print('- training_validation_curves.png')
print('- test_metric_comparison.png')
print('- test_summary.csv / test_summary.json')
print('- test_metrics_per_image.csv')
print('- worst_test_failures.png')


## 7. Final visual gate and export

The middle columns below are the honest comparison. Do not choose the refined model only
because training completed. Accept it only if it improves structure, realism, faces, and
general scenes without colour artifacts. The final test set remains untouched.


In [ ]:
BEST_PATH = RUN_ROOT / 'best.pt'
assert BEST_PATH.is_file(), 'Run the training cell first.'
best_state = torch.load(BEST_PATH, map_location='cpu', weights_only=True)
ema_refiner.load_state_dict(best_state['ema_refiner'], strict=True)
ema_refiner.to(DEVICE).eval()
render_gate(ema_refiner, best_state['step'], 'FINAL_VISUAL_GATE.png')

manifest = {
    'format': 'sketch2photo-focused-refiner-v1',
    'direction': 'sketch_to_photo',
    'quality_status': 'Requires human visual acceptance; exact colours are not recoverable from monochrome input.',
    'best_step': best_state['step'],
    'validation': best_state['validation'],
    'checkpoint': 'best.pt',
    'coarse_source': str(COARSE_CHECKPOINT),
    'photo_to_pencil_status': 'Untouched; continue using the previously accepted checkpoint.',
}
atomic_json(RUN_ROOT / 'manifest.json', manifest)

export_root = RUN_ROOT / 'export'
export_root.mkdir(exist_ok=True)
shutil.copy2(BEST_PATH, export_root / 'best.pt')
shutil.copy2(RUN_ROOT / 'manifest.json', export_root / 'manifest.json')
shutil.copy2(RUN_ROOT / 'recipe.json', export_root / 'recipe.json')
shutil.copy2(RUN_ROOT / 'FINAL_VISUAL_GATE.png', export_root / 'FINAL_VISUAL_GATE.png')
for report_path in REPORT_ROOT.iterdir():
    if report_path.is_file():
        shutil.copy2(report_path, export_root / report_path.name)
archive = shutil.make_archive(str(RUN_ROOT / 'focused_sketch_to_photo_export'), 'zip', export_root)

print('Review:', RUN_ROOT / 'FINAL_VISUAL_GATE.png')
print('Drive export:', archive)
print('Training/resume state remains in:', RUN_ROOT)
print('Do not replace the application model until the visual gate is acceptable.')


## Resume and completion checklist

- If Colab reaches its limit: reconnect, enable a GPU, and run every cell again. The
  training cell resumes automatically from the exact saved update.
- Do not delete `last_training_state.pt` while training is incomplete.
- `best.pt` is validation-selected and may come from an earlier update than the last one.
- Review `FINAL_VISUAL_GATE.png` before integrating the model into VS Code.
- Keep the old photo→pencil checkpoint. This notebook never changes it.
- For the report, describe this as a **coarse-to-fine residual refiner trained with a
  conditional multi-scale PatchGAN objective**, and disclose that the coarse U-Net was
  warm-started from earlier training.
